# FoV figure: 21-cm fields of view, survey footprints and simulation boxes on an EOS26 slice

Makes the figure of `plot_fov_zoom.py figure`: a slice of the rect lightcone at constant redshift (150 MHz, z = 8.47 by
default) with the fields of view of SKA-Low, LOFAR and HERA, the SPHEREx deep field and the 500, 100 and 20 cMpc
simulation boxes, and a zoom panel on the centre with the footprints of the other surveys (CCAT/FYST
EoR-Spec, JWST COSMOS-3D, Roman UDF) over three bands (the figure's quantity, then two of $x_{\rm HI}$, $\delta T_{\rm b}$ and density).

The plotting code is `make_figure` and `draw_figure_footprints` in `plot_fov_zoom.py`, and the survey footprints and
simulation boxes are its `FOOTPRINTS` and `SIM_BOXES`. The settings below are its command-line options
(`python plot_fov_zoom.py figure --help`); the notebook runs it and shows the PNG (a PDF is written next to it).
With `SIM_BOXES = False` the simulation boxes are left out (`--no-sim-boxes`) and `_nosim` is added to the file name.

In [ ]:
import os
import sys
from pathlib import Path

os.environ.setdefault("EOS26_TEMPLATE", "EOS26_earlyEoR")  # the production run, read when plot_fov_zoom is imported
HERE = Path.cwd()  # postprocess/
sys.path.insert(0, str(HERE))
import plot_fov_zoom as pfz  # noqa: E402

## Settings

In [ ]:
PRODUCTION = True  # False: the HII_DIM = 200 test run
QUANTITY = "neutral_fraction"  # e.g. brightness_temp, density; the first band of the zoom panel
SIM_BOXES = False  # True: with the 500, 100 and 20 cMpc simulation boxes; False: without (--no-sim-boxes, ..._nosim)
RUN = Path(os.environ["EOS26_TEMPLATE"]).name.removesuffix(".toml")
# slice frequency [MHz]: 150 (z = 8.47), and for EOS26_earlyEoR 120.7 (z = 10.77), where its mean spin temperature
# (28 K) is below the CMB's (32 K): absorption, with emission where the gas is heated (at 128.1 MHz, z = 10.09, it is
# nearly all emission); the fields of view, footprints and survey names are those of 150 MHz in both
FREQ = 120.7 if RUN == "EOS26_earlyEoR" else 150.0
# more options, e.g. "--inset-bands" with no quantity (one band in the panel);
# "--band-galaxies", "3" (galaxies drawn in each band of the panel);
# "--fig-width", "7" (inches: sets the size of the text relative to the figure)
OPTIONS = ["--halos", "auto", "--freq", str(FREQ), "--overlay-freq", "150", "--deg-axis"]
OUT = HERE.parent / "plots" / f"fov_{RUN}_z{1420.405751 / FREQ - 1:.2f}_{QUANTITY}"  # without suffix: .png and .pdf
if not SIM_BOXES:
    OPTIONS += ["--no-sim-boxes"]
    OUT = OUT.with_name(OUT.name + "_nosim")

## [C II] placeholder

Until the [C II] lightcone exists, the third band is the star formation rate density (`sfrd_acg`; [C II] traces
star formation) in the colour map of [C II], with its colour bar labelled SFRD and "[C II] placeholder" in red on
the band. With the real [C II], delete this section and give its quantity in `--inset-bands`.

In [ ]:
import functools

pfz.SPEC["sfrd_acg"] = ("[C II] placeholder", "cividis", "log")
pfz.CBAR_LABEL["sfrd_acg"] = r"${\rm SFRD}$"  # what the band shows
OPTIONS += ["--inset-bands", "brightness_temp", "sfrd_acg"]
# the original function, also when this cell runs again
draw = getattr(pfz.draw_figure_footprints, "__wrapped__", pfz.draw_figure_footprints)


@functools.wraps(draw)
def draw_with_note(fig, *args, **kwargs):  # the panel is the figure's second axes, the third band its right third
    draw(fig, *args, **kwargs)
    fig.axes[1].text(5 / 6, 0.5, "[C II] placeholder", transform=fig.axes[1].transAxes, rotation=90, color="red",
                     fontsize=16, fontweight="bold", ha="center", va="center", zorder=6,
                     path_effects=pfz.outlined(3.0, "black", 0.85))


pfz.draw_figure_footprints = draw_with_note

## Figure

In [ ]:
argv = ["figure", "--quantity", QUANTITY, "--out", str(OUT)] + (["--production"] if PRODUCTION else []) + OPTIONS
pfz.main(argv)
try:  # in Jupyter, show it (run_notebook.py runs the notebook without IPython)
    from IPython.display import Image, display

    display(Image(filename=f"{OUT}.png"))
except ImportError:
    pass